# Iter-3 CCRG Re-Run — Decisive Analysis Layer (M1 / M2 / M3)

**Two-Track Counterfactual Co-Response Grouping (CCRG) on first-letter spelling.**

This artifact re-runs the iter-2 two-track CCRG pipeline (a frozen **Gemma-Scope L12 / 16k JumpReLU
SAE** over **gemma-2-2b**) and surgically adds **three fixes** that decide whether the contribution is
genuine *two-track SELECTION* or merely *cover-based eligibility + max-pooling*:

- **M1 — Random-Eligible-k (RE-k) baseline (decisive).** Draw `k = |unit|` latents *uniformly at random*
  from the cover-eligible set and max-pool them identically to the unit. Because RE-k differs from the
  unit **only** in the membership/SELECTION criterion, `unit − RE-k` isolates *selection* from
  *eligibility + pooling*. The single most decisive number is **`frac_rek_ge_unit`** — the fraction of
  random eligible pools whose test-fold AUC matches/beats the unit (a one-sided permutation p-value).
- **M2 — AUC points + AUC-difference CIs.** Threshold-free held-out test-fold AUC, bootstrap
  AUC-difference CIs (B ≥ 10,000), a pooled-across-letters meta-analysis, and a **Youden** (TPR−FPR)
  accuracy table that does **not** collapse to predict-all-positive.
- **M3 — Verdict from the stated falsifier.** `primary_endpoint` is computed transparently from
  *E1 AND unit-AUC-significantly-above-BOTH-(h)-AND-(RE-k) on ≥ 3/5 letters*; E2 is never silently dropped.

### What this demo runs

The heavy stages — the **gemma-2-2b forward pass** and **Gemma-Scope SAE encoding** on a GPU — produced
the per-example held-out predictions and the AUC tables that we load here from `mini_demo_data.json`.
This notebook reproduces the **decisive analysis layer (M1 / M2 / M3)** on that output, using
`method.py`'s **own statistics functions verbatim**, so it runs in seconds on a CPU. We:

1. rebuild the per-example test-fold table for the primary letter **L** and recompute the **Youden
   accuracy** for every method (exactly reproducing the saved `accuracy_youden`), adding **paired
   bootstrap accuracy-difference CIs** and **McNemar/Holm** tests (M2);
2. display the precomputed **AUC points + AUC-difference CIs** (M2);
3. show the **RE-k draw distribution** and `frac_rek_ge_unit` across all five letters (M1);
4. recompute the **`primary_endpoint` verdict** from the stated falsifier and confirm it matches the
   saved verdict (M3).


In [ ]:
# --- Dependencies -----------------------------------------------------------------------------
# Every package used by the analysis layer (numpy, scipy, scikit-learn, matplotlib, statsmodels)
# is pre-installed on Colab. On Colab we MUST NOT reinstall them (it corrupts the loaded C
# extensions); locally we install them at Colab's exact versions so the environment matches.
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1',
         'matplotlib==3.10.0', 'statsmodels==0.14.6')


In [ ]:
# --- Imports (from method.py, plus matplotlib for the demo visualisation) ---------------------
import os, json, time, warnings
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Constants carried verbatim from method.py
SEED = 1234                               # method.py SEED (drives the paired bootstrap RNG)
LETTERS_ALL = ["L", "O", "T", "I", "D"]   # the five first-letter testbeds
PRIMARY = "L"

# method.py logging helper (verbatim)
T0 = time.time()
def log(msg):
    t = time.strftime("%H:%M:%S")
    print(f"[{t}] {msg}", flush=True)
def el():
    return f"{time.time()-T0:6.1f}s"


In [ ]:
# --- Data loading: GitHub URL (Colab) with local fallback (now) -------------------------------
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ee30c-catching-silent-feature-absorption-in-fr/main/round-3/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")


In [ ]:
data = load_data()
meta = data["metadata"]
per_letter = meta["per_letter_summary"]          # slim precomputed per-letter tables (M1/M2/M3 inputs)
saved_verdicts = meta["verdicts"]                # the verdict produced by the full GPU run

log(f"{el()} loaded mini_demo_data.json")
print("config:", json.dumps(meta["config"]["thresholds"], indent=0))
print("letters with summary tables:", list(per_letter.keys()))
print("dataset groups:", [(g['dataset'], len(g['examples'])) for g in data['datasets']])
print("saved primary_endpoint:", saved_verdicts["primary_endpoint"])


In [ ]:
# --- Demo configuration -----------------------------------------------------------------------
# Start at the ABSOLUTE MINIMUM and scale up (the analysis is pure CPU and finishes in seconds).
DEMO_LETTER  = "L"      # primary letter; its per-example test-fold rows are bundled in the demo data
B_BOOTSTRAP  = 10000    # paired accuracy-difference bootstrap draws.
                        # method.py default (b_gap) = 10000.  Minimum that still produces a CI: ~200.
BASELINES    = ["a", "b", "c", "h", "REk"]   # methods the two-track unit is compared against

# Human-readable names for the methods (for tables / plots)
METHOD_LABEL = {
    "unit": "two-track unit",
    "a":    "(a) single best latent",
    "b":    "(b) top-k by train AUC",
    "c":    "(c) top-k corr-pruned",
    "h":    "(h) oracle attribution",
    "REk":  "(RE-k) random-eligible-k",
}
print(f"DEMO_LETTER={DEMO_LETTER}  B_BOOTSTRAP={B_BOOTSTRAP}  baselines={BASELINES}")


## A. Rebuild the per-example held-out test-fold table

Each bundled example row is one held-out **test-fold** instance for letter `L` (33 `on`-words that start
with L, 33 surface-matched `off`-words). Every row carries the true label
(`metadata_label_starts_with_target`) and each method's **Youden-thresholded** prediction
(`predict_unit/a/b/c/h/REk`) — exactly what `method.py` writes out at `run_c1` line 1305. From these we
build per-method **correctness vectors** (`prediction == true label`), the substrate for the M2 accuracy
analysis.

In [ ]:
# Pull the held-out test-fold rows for the demo letter
group = next(g for g in data["datasets"] if g["dataset"] == f"first_letter_spelling_{DEMO_LETTER}")
rows = group["examples"]

y_true = np.array([int(r["metadata_label_starts_with_target"]) for r in rows])
all_names = ["unit"] + BASELINES

# Per-method correctness vector: predict_<name> == true label (predictions are "0"/"1" strings)
correctness = {name: np.array([int(int(r[f"predict_{name}"]) == int(r["metadata_label_starts_with_target"]))
                               for r in rows])
               for name in all_names}

log(f"{el()} {DEMO_LETTER}: {len(rows)} test-fold rows  "
    f"({int((y_true==1).sum())} on / {int((y_true==0).sum())} off)")
print("sub-context roles:", dict(Counter(r['metadata_role'] for r in rows)))
print("\nfirst 5 rows:")
for r in rows[:5]:
    print(f"  {r['metadata_role']:3s}  label={r['metadata_label_starts_with_target']}  "
          f"word={r['metadata_sub_context']:<10s} "
          f"unit={r['predict_unit']} h={r['predict_h']} REk={r['predict_REk']}  | {r['input']!r}")


## B. M2 — Youden accuracy table + paired accuracy-difference CIs (recomputed live)

We recompute, for every method, the **comparison-matched Youden accuracy** straight from the saved
predictions — it reproduces the GPU run's `accuracy_youden` exactly — and add **paired bootstrap
accuracy-difference CIs** (`unit − baseline`) and **exact McNemar** tests (Holm-corrected). The three
statistics functions below are copied **verbatim** from `method.py` (`paired_bootstrap_diff`,
`mcnemar_p`, `holm`).

In [ ]:
# ---- statistics helpers copied VERBATIM from method.py ---------------------------------------
def paired_bootstrap_diff(a, b, B=10000, rng=None):
    """CI on mean(a)-mean(b), paired (same indices). a,b arrays of per-example values."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    n = len(a)
    if n == 0:
        return {"diff": 0.0, "ci_lo": 0.0, "ci_hi": 0.0, "excl_0": False, "n": 0}
    rng = rng or np.random.default_rng(SEED)
    idx = rng.integers(0, n, size=(B, n))
    d = a[idx].mean(1) - b[idx].mean(1)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"diff": float(a.mean() - b.mean()), "ci_lo": float(lo), "ci_hi": float(hi),
            "excl_0": bool(lo > 0 or hi < 0), "n": int(n)}


def mcnemar_p(correct_a, correct_b):
    """Exact McNemar on paired binary correctness vectors."""
    from statsmodels.stats.contingency_tables import mcnemar
    a = np.asarray(correct_a).astype(bool); b = np.asarray(correct_b).astype(bool)
    n01 = int(np.sum(a & ~b)); n10 = int(np.sum(~a & b))
    table = [[int(np.sum(a & b)), n01], [n10, int(np.sum(~a & ~b))]]
    try:
        res = mcnemar(table, exact=True)
        return float(res.pvalue), n01, n10
    except Exception:
        return 1.0, n01, n10


def holm(pvals):
    from statsmodels.stats.multitest import multipletests
    if not pvals:
        return [], []
    rej, p_adj, _, _ = multipletests(pvals, method="holm")
    return [bool(x) for x in rej], [float(x) for x in p_adj]


In [ ]:
# ---- per-method Youden accuracy (live) vs the saved accuracy_youden --------------------------
pm = per_letter[DEMO_LETTER]["C1"]["per_method"]      # saved per-method table from the GPU run

print(f"=== M2 Youden accuracy — letter {DEMO_LETTER} (n={len(rows)}) ===")
print(f"{'method':<28s} {'acc(live)':>9s} {'acc(saved)':>10s} {'AUC(saved)':>10s}")
for name in all_names:
    acc_live = float(correctness[name].mean())
    acc_saved = pm[name]["accuracy_youden"]
    auc_saved = pm[name]["AUC"]
    flag = "  <-- reproduces" if abs(acc_live - acc_saved) < 1e-9 else "  !! MISMATCH"
    print(f"{METHOD_LABEL[name]:<28s} {acc_live:9.4f} {acc_saved:10.4f} {auc_saved:10.4f}{flag}")


In [ ]:
# ---- paired accuracy-difference CIs (unit - baseline) + McNemar/Holm (M2 block, live) --------
rng = np.random.default_rng(SEED)
cis, pv, keys = {}, [], []
for bl in BASELINES:
    ci = paired_bootstrap_diff(correctness["unit"], correctness[bl], B=B_BOOTSTRAP, rng=rng)
    cis[f"unit_vs_{bl}"] = ci
    p, n01, n10 = mcnemar_p(correctness["unit"], correctness[bl]); pv.append(p); keys.append(bl)
rej, padj = holm(pv)

print(f"=== M2 paired accuracy-difference (unit - baseline), B={B_BOOTSTRAP} ===")
print(f"{'comparison':<26s} {'diff':>7s} {'95% CI':>20s} {'excl0':>6s} {'McNemar p(Holm)':>16s}")
for i, bl in enumerate(keys):
    ci = cis[f"unit_vs_{bl}"]
    print(f"{'unit vs '+METHOD_LABEL[bl]:<26s} {ci['diff']:+7.3f} "
          f"[{ci['ci_lo']:+6.3f},{ci['ci_hi']:+6.3f}] {str(ci['excl_0']):>6s} "
          f"{padj[i]:16.4f}{'  *' if rej[i] else ''}")


## C. M2 — AUC points + AUC-difference CIs (from the GPU run)

The threshold-free **held-out test-fold AUC** and the **bootstrap AUC-difference CIs** (whole
content-flip pair cluster resampling, `B = 10,000`) are the primary M2 endpoints. They are computed on
the raw SAE scores during the GPU run; we display the saved table for the demo letter. Note the decisive
pattern on `L`: the unit **significantly beats RE-k** (`sig_unit_better = True`) but **ties the oracle
attribution baseline (h)** (CI includes 0) — exactly the pre-registered expectation.

In [ ]:
ad = per_letter[DEMO_LETTER]["C1"]["auc_diff"]
print(f"=== M2 AUC-difference CIs — unit vs each baseline, letter {DEMO_LETTER} ===")
print(f"{'comparison':<26s} {'auc_unit':>8s} {'auc_x':>7s} {'diff':>7s} {'95% CI':>20s} {'sig_unit_better':>15s}")
for bl in BASELINES:
    cd = ad[f"unit_vs_{bl}"]
    print(f"{'unit vs '+METHOD_LABEL[bl]:<26s} {cd['auc_unit']:8.3f} {cd['auc_x']:7.3f} "
          f"{cd['diff']:+7.3f} [{cd['ci_lo']:+6.3f},{cd['ci_hi']:+6.3f}] {str(cd['sig_unit_better']):>15s}")


## D. M1 — Random-Eligible-k (RE-k) baseline (the decisive control)

For every letter, `k = |unit|` latents are drawn uniformly at random from the **cover-eligible** set
and max-pooled identically to the unit. The only thing that differs from the two-track unit is the
**SELECTION** criterion, so `unit − RE-k` isolates selection from eligibility + pooling. The headline
statistic **`frac_rek_ge_unit`** is the fraction of random eligible pools whose test-fold AUC
matches/beats the unit — a one-sided permutation p-value.

In [ ]:
print("=== M1 RE-k draw distribution vs unit AUC (per letter) ===")
print(f"{'letter':>6s} {'unit AUC':>8s} {'RE-k p50':>8s} {'RE-k p95':>8s} {'RE-k max':>8s} "
      f"{'k':>3s} {'frac_rek>=unit':>14s}")
for L in LETTERS_ALL:
    rd = per_letter[L]["C1"]["rek_distribution"]
    print(f"{L:>6s} {rd['unit_test_AUC']:8.3f} {rd['draw_auc_p50']:8.3f} {rd['draw_auc_p95']:8.3f} "
          f"{rd['draw_auc_max']:8.3f} {rd['k']:>3d} {rd['frac_rek_ge_unit']:14.3f}")
print("\nOn primary L: frac_rek_ge_unit =",
      per_letter['L']['C1']['rek_distribution']['frac_rek_ge_unit'],
      "(only ~0.9% of random eligible pools reach the two-track unit's AUC).")


## E. M3 — Verdict from the stated falsifier (recomputed live)

The `primary_endpoint` is decided by the stated falsifier: **E1** (≥ 4/5 letters) **AND** the unit's
test-fold AUC being **significantly above BOTH the oracle attribution (h) AND the random-eligible-k
(RE-k)** on **≥ 3/5** letters. The verdict logic below is copied **verbatim** from `method.py`
(`build_verdicts` / `_sig`); we recompute it from the per-letter tables and confirm it matches the saved
verdict.

In [ ]:
# ---- verdict logic copied VERBATIM from method.py (build_verdicts core + _sig) ---------------
def _sig(res, comp):
    """sig_unit_better (CI lower bound > 0) for a C1 AUC-difference comparison."""
    return bool(res.get("C1", {}).get("auc_diff", {}).get(comp, {}).get("sig_unit_better", False))


def build_verdict(per_letter):
    letters = [l for l in LETTERS_ALL if l in per_letter]
    per_E1 = {l: bool(per_letter[l].get("E1", {}).get("E1_PASS")) for l in letters}
    per_E2 = {l: bool(per_letter[l].get("E2", {}).get("E2_PASS")) for l in letters}
    sel_pass = {l: bool(_sig(per_letter[l], "unit_vs_h") and _sig(per_letter[l], "unit_vs_REk")) for l in letters}
    elig_only = {l: bool(_sig(per_letter[l], "unit_vs_h") and not _sig(per_letter[l], "unit_vs_REk")) for l in letters}
    n_E1 = sum(1 for l in letters if per_E1[l]); E1_overall = bool(n_E1 >= 4)
    n_selection = sum(1 for l in letters if sel_pass[l])
    n_eligibility = sum(1 for l in letters if elig_only[l])
    if E1_overall and n_selection >= 3:
        endpoint = "ABSORPTION_REPAIR_SELECTION_CONFIRMED"
    elif (n_selection + n_eligibility) >= 3 and n_selection < 3:
        endpoint = "REFRAMED_TO_ELIGIBILITY_POOLING"
    else:
        endpoint = "SELECTION_NOT_ESTABLISHED"
    return {"primary_endpoint": endpoint, "E1_overall": E1_overall, "n_E1_pass": n_E1,
            "n_selection": n_selection, "n_eligibility": n_eligibility,
            "per_letter_E1": per_E1, "per_letter_E2": per_E2,
            "per_letter_sel_pass": sel_pass, "per_letter_elig_only": elig_only}


v = build_verdict(per_letter)
print("per-letter SELECTION test (unit significantly above BOTH h AND RE-k):")
for L in LETTERS_ALL:
    print(f"  {L}: E1={v['per_letter_E1'][L]!s:5s} "
          f"sig_vs_h={_sig(per_letter[L],'unit_vs_h')!s:5s} "
          f"sig_vs_REk={_sig(per_letter[L],'unit_vs_REk')!s:5s} -> selection={v['per_letter_sel_pass'][L]}")
print(f"\nn_E1_pass={v['n_E1_pass']}/5  n_selection={v['n_selection']}/5  -> {v['primary_endpoint']}")
print("recomputed == saved :", v["primary_endpoint"] == saved_verdicts["primary_endpoint"])

pooled = meta["pooled_across_letters"]
for comp in ("unit_vs_h", "unit_vs_REk"):
    sb = pooled[comp]["stratified_bootstrap"]
    print(f"\npooled {comp}: diff={sb['diff']:+.3f} CI[{sb['ci_lo']:+.3f},{sb['ci_hi']:+.3f}] "
          f"sig_unit_better={sb['sig_unit_better']}")


## F. Visualisation — the decisive picture

Left: per-method held-out AUC for the demo letter (the two-track unit highlighted). Middle: per-letter
**unit AUC vs the RE-k draw distribution** (the p05–p95 band of random eligible pools) — the unit sits
above the random band everywhere. Right: the per-letter `frac_rek_ge_unit` (lower = stronger selection
signal). The recomputed verdict is printed underneath.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# --- (1) per-method AUC bar chart for the demo letter ---
names = ["unit"] + BASELINES
aucs = [pm[n]["AUC"] for n in names]
colors = ["#1f77b4" if n == "unit" else "#9ecae1" for n in names]
axes[0].bar([METHOD_LABEL[n].replace("two-track ", "") for n in names], aucs, color=colors)
axes[0].axhline(0.5, ls="--", c="grey", lw=1)
axes[0].set_ylim(0.4, 1.0); axes[0].set_ylabel("held-out test-fold AUC")
axes[0].set_title(f"M2 — per-method AUC (letter {DEMO_LETTER})")
axes[0].tick_params(axis="x", rotation=35)
for i, a in enumerate(aucs):
    axes[0].text(i, a + 0.008, f"{a:.2f}", ha="center", fontsize=8)

# --- (2) per-letter unit AUC vs RE-k draw band ---
xs = np.arange(len(LETTERS_ALL))
unit_aucs = [per_letter[L]["C1"]["rek_distribution"]["unit_test_AUC"] for L in LETTERS_ALL]
rek_p50   = [per_letter[L]["C1"]["rek_distribution"]["draw_auc_p50"] for L in LETTERS_ALL]
rek_lo    = [per_letter[L]["C1"]["rek_distribution"]["draw_auc_p05"] for L in LETTERS_ALL]
rek_hi    = [per_letter[L]["C1"]["rek_distribution"]["draw_auc_p95"] for L in LETTERS_ALL]
yerr = np.array([np.array(rek_p50) - np.array(rek_lo), np.array(rek_hi) - np.array(rek_p50)])
axes[1].errorbar(xs, rek_p50, yerr=yerr, fmt="o", c="#9467bd", capsize=4, label="RE-k draws (p05–p50–p95)")
axes[1].plot(xs, unit_aucs, "D", c="#1f77b4", ms=9, label="two-track unit")
axes[1].set_xticks(xs); axes[1].set_xticklabels(LETTERS_ALL)
axes[1].axhline(0.5, ls="--", c="grey", lw=1)
axes[1].set_ylim(0.4, 1.0); axes[1].set_ylabel("test-fold AUC")
axes[1].set_title("M1 — unit vs random-eligible-k pools"); axes[1].legend(fontsize=8)

# --- (3) frac_rek_ge_unit per letter ---
frac = [per_letter[L]["C1"]["rek_distribution"]["frac_rek_ge_unit"] for L in LETTERS_ALL]
axes[2].bar(LETTERS_ALL, frac, color="#d62728")
axes[2].axhline(0.05, ls="--", c="grey", lw=1, label="p = 0.05")
axes[2].set_ylabel("frac_rek_ge_unit  (one-sided p)")
axes[2].set_title("M1 — decisive permutation p-value"); axes[2].legend(fontsize=8)
for i, fr in enumerate(frac):
    axes[2].text(i, fr + 0.005, f"{fr:.3f}", ha="center", fontsize=8)

plt.tight_layout(); plt.show()

print("=" * 72)
print(f"M3 PRIMARY ENDPOINT (recomputed): {v['primary_endpoint']}")
print(f"  E1 passes on {v['n_E1_pass']}/5 letters; unit beats BOTH h AND RE-k on "
      f"{v['n_selection']}/5 letters.")
print(f"  matches saved verdict: {v['primary_endpoint'] == saved_verdicts['primary_endpoint']}")
print("=" * 72)
